# 中国银行 (601988.SH) 量化分析 Notebook

**流程：** 读取本地 CSV → 计算技术指标 → 画K线图 → 基本面+技术面分析

**作者：** BA-Quant &nbsp;|&nbsp; **日期：** 2026-07-01

## 1. 环境准备

In [ ]:
!pip install pandas numpy plotly -q

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("✅ 环境准备完成")

## 2. 读取本地 CSV 数据

直接从同目录下的 **601988_daily.csv** 读取中国银行近一年日线数据。

In [ ]:
df = pd.read_csv('601988_daily.csv', parse_dates=['date'])

print(f"✅ 读取 {len(df)} 条交易数据")
print(f"日期范围：{df['date'].min().strftime('%Y-%m-%d')} ~ {df['date'].max().strftime('%Y-%m-%d')}")
print(f"字段列表：{list(df.columns)}")
df.head(10)

In [ ]:
# 数据概览
df.describe()

## 3. 计算技术指标

MA（移动平均线）、BOLL（布林带）、RSI（相对强弱）、MACD：

In [ ]:
# 移动平均线
for p in [5, 10, 20, 60, 120]:
    df[f'ma{p}'] = df['close'].rolling(window=p).mean()

# 布林带 (20, 2)
df['boll_mid'] = df['close'].rolling(20).mean()
boll_std = df['close'].rolling(20).std()
df['boll_upper'] = df['boll_mid'] + 2 * boll_std
df['boll_lower'] = df['boll_mid'] - 2 * boll_std

# RSI(14)
def calc_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False).mean()
    rs = avg_gain / avg_loss
    return (100 - 100 / (1 + rs)).round(2)

df['rsi'] = calc_rsi(df['close'], 14)

# MACD
ema12 = df['close'].ewm(span=12, adjust=False).mean()
ema26 = df['close'].ewm(span=26, adjust=False).mean()
df['dif'] = (ema12 - ema26).round(4)
df['dea'] = df['dif'].ewm(span=9, adjust=False).mean().round(4)
df['macd_bar'] = (2 * (df['dif'] - df['dea'])).round(4)

print("✅ 技术指标计算完成")
df[['date','close','ma5','ma10','ma20','rsi','dif','dea']].tail(10)

## 4. 绘制 K 线图（Plotly 交互式）

### 4.1 K 线图 + 均线 + 布林带 + 成交量 + MACD

In [ ]:
fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.03,
    row_heights=[0.55, 0.2, 0.25],
    subplot_titles=('K线图 + 均线 + 布林带', '成交量', 'MACD')
)

# 子图1：K线 + 均线 + 布林带
fig.add_trace(go.Candlestick(
    x=df['date'], open=df['open'], high=df['high'],
    low=df['low'], close=df['close'], name='K线',
    increasing_line_color='#ef4444', decreasing_line_color='#22c55e'
), row=1, col=1)

for name, color in [('ma5','#f59e0b'),('ma10','#06b6d4'),('ma20','#a855f7'),('ma60','#ec4899')]:
    fig.add_trace(go.Scatter(x=df['date'], y=df[name], mode='lines',
        name=name.upper(), line=dict(width=1, color=color)), row=1, col=1)

for band in ['boll_upper', 'boll_lower']:
    fig.add_trace(go.Scatter(x=df['date'], y=df[band], mode='lines', name='BOLL',
        line=dict(width=0.8, color='rgba(59,130,246,0.4)', dash='dash'),
        showlegend=False), row=1, col=1)

# 子图2：成交量
vol_colors = ['#ef4444' if df['close'].iloc[i] >= df['open'].iloc[i] else '#22c55e'
              for i in range(len(df))]
fig.add_trace(go.Bar(x=df['date'], y=df['volume'], name='成交量',
    marker_color=vol_colors, showlegend=False), row=2, col=1)

# 子图3：MACD
fig.add_trace(go.Scatter(x=df['date'], y=df['dif'], mode='lines',
    name='DIF', line=dict(width=1, color='#f59e0b')), row=3, col=1)
fig.add_trace(go.Scatter(x=df['date'], y=df['dea'], mode='lines',
    name='DEA', line=dict(width=1, color='#06b6d4')), row=3, col=1)
macd_colors = ['#ef4444' if v >= 0 else '#22c55e' for v in df['macd_bar']]
fig.add_trace(go.Bar(x=df['date'], y=df['macd_bar'], name='MACD',
    marker_color=macd_colors, showlegend=False), row=3, col=1)

fig.update_layout(
    title='中国银行 (601988.SH) 近一年 K 线分析',
    template='plotly_dark', height=900,
    xaxis_rangeslider_visible=False, hovermode='x unified',
    legend=dict(orientation='h', yanchor='top', y=1.12, xanchor='left', x=0),
    margin=dict(t=60, b=20, l=60, r=40)
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridcolor='rgba(128,128,128,0.15)')
fig.show()
print("✅ K线图绘制完成")

### 4.2 RSI 指标图

In [ ]:
fig_rsi = go.Figure()
fig_rsi.add_trace(go.Scatter(x=df['date'], y=df['rsi'], mode='lines',
    name='RSI(14)', line=dict(width=1.5, color='#a855f7'),
    fill='tozeroy', fillcolor='rgba(168,85,247,0.08)'))
fig_rsi.add_hline(y=70, line_dash='dash', line_color='rgba(239,68,68,0.5)',
    annotation_text='超买 70', annotation_position='top right')
fig_rsi.add_hline(y=30, line_dash='dash', line_color='rgba(34,197,94,0.5)',
    annotation_text='超卖 30', annotation_position='bottom right')
fig_rsi.add_hline(y=50, line_dash='dot', line_color='rgba(128,128,128,0.25)')
fig_rsi.update_layout(title='RSI(14) 指标', template='plotly_dark', height=350,
    margin=dict(t=40, b=20, l=60, r=40), yaxis=dict(range=[0, 100]))
fig_rsi.show()
print("✅ RSI 图绘制完成")

## 5. 基本面数据分析

In [ ]:
latest = df.iloc[-1]
prev = df.iloc[-2]
max_52w = df['high'].max()
min_52w = df['low'].min()
avg_vol_20 = df['volume'].tail(20).mean()

print("=" * 55)
print("  中国银行 (601988.SH) 量价数据汇总")
print("=" * 55)
print(f"  最新收盘价：    ¥{latest['close']:.2f}")
print(f"  日涨跌幅：      {((latest['close']-prev['close'])/prev['close']*100):+.2f}%")
print(f"  52周最高价：    ¥{max_52w:.2f}")
print(f"  52周最低价：    ¥{min_52w:.2f}")
print(f"  近20日均量：    {avg_vol_20/10000:.0f} 万手")
print(f"  RSI(14)：       {latest['rsi']:.2f}")
print(f"  MA20：          ¥{latest['ma20']:.2f}")
print(f"  MA60：          ¥{latest['ma60']:.2f}")
print()
print("=" * 55)
print("  基本面指标")
print("=" * 55)
for k, v in {
    '2025年营收':'6,599亿 (+4.28% YoY)',
    '2025年归母净利':'2,430亿 (+2.18% YoY)',
    '2026Q1营收':'1,788亿 (+8.4% YoY)',
    '2026Q1归母净利':'566亿 (+4.2% YoY)',
    'ROE (2025)':'8.94%',
    '净息差 (2025)':'1.26%',
    '不良率':'1.23% (六大行最低)',
    '当前PB':'≈0.62x (破净)',
    '全年股息率':'≈4.0%',
    '每股派息':'0.2263元 (分红率30%)',
}.items():
    print(f"  {k}：{v}")

## 6. 总结

| 步骤 | 内容 |
|------|------|
| 1. 环境准备 | pandas / numpy / plotly |
| 2. 数据读取 | 读取同目录 `601988_daily.csv` |
| 3. 技术指标 | MA5/10/20/60、BOLL(20,2)、RSI(14)、MACD(12,26,9) |
| 4. K线图 | Plotly 交互式三面板：K线+均线+布林 / 成交量 / MACD |
| 5. RSI图 | RSI(14) 超买超卖分析 |
| 6. 基本面 | 营收/净利/ROE/息差/不良率/PB/股息汇总 |

**扩展建议：** 修改 `TS_CODE` 可分析任意 A 股；添加量化策略回测；GitHub Actions 定时运行生成每日报告。